# Text Cleaning
Hai strategy cleaning khác nhau:
1. **Strict Cleaning**: TF-IDF, Word2Vec, FastText, GloVe
2. **Loose Cleaning**: PhoBERT-v2, Bert4News, ChatGPT

In [14]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

# Set pandas options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 1. Load Data

In [15]:
# Load original data
df = pd.read_csv('../../data/raw/data.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSample text:")
print(df['post_message'].iloc[0][:200])

Dataset shape: (4736, 26)
Columns: ['id', 'user_name', 'timestamp_post', 'post_message', 'label', 'num_char', 'num_emoji', 'num_url', 'num_hashtag', 'num_post', 'num_real', 'num_fake', 'post_ratio', 'num_like', 'num_cmt', 'num_share', 'pixel', 'image', 'num_image', 'hour', 'weekday', 'day', 'month', 'year', 'time', 'min']

Sample text:
THĂNG CẤP_BẬC HÀM ĐỐI_VỚI 2 CÁN_BỘ , CHIẾN_SỸ HY_SINH Ở ĐÀ_NẴNG Ngày 3/4 , Đại_tướng Tô_Lâm , Bộ_trưởng Bộ Công_an đã ký quyết_định số 2398 / QĐ-BCA-X01 thăng cấp_bậc hàm từ Đại_úy lên Thiếu_tá đối_vớ


## 2. Define Cleaning Functions

In [16]:
def strict_clean(text):
    """
    Strict cleaning for TF-IDF, Word2Vec, FastText, GloVe
    Removes everything unnecessary, aggressive preprocessing
    """
    if pd.isna(text) or text == '':
        return ''
    
    # 1. Convert to lowercase
    text = text.lower()
    
    text = re.sub(r'http\S+|www\S+', '[URL]', text)  

    text = re.sub(r'<\s*url\s*>', '[URL]', text)

    text = re.sub(r'\S+@\S+', '[EMAIL]', text)             
    
    # 6. Remove hashtags symbols but keep text
    text = re.sub(r'#(\w+)', r'\1', text)  # #word → word
    
    # 7. Remove special characters (keep Vietnamese diacritics)
    text = re.sub(r'[^\w\s\u0100-\u017f\u0180-\u024f]', '', text, flags=re.UNICODE)
    
    # 8. Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # 9. Remove extra spaces
    text = ' '.join(text.split())
    
    return text


def loose_clean(text):
    """
    Loose cleaning for PhoBERT-v2, Bert4News, ChatGPT
    Minimal preprocessing - let BERT/embedding model handle
    """
    if pd.isna(text) or text == '':
        return ''
    
    # 1. Remove URLs (replace with placeholder)
    text = re.sub(r'http\S+|www\S+', '[URL]', text)
    # Remove generic < url > placeholder
    text = re.sub(r'<\s*url\s*>', '[URL]', text)
    
    # 2. Remove email (replace with placeholder)
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    
    # 3. Remove HTML tags but keep content
    text = re.sub(r'<[^>]+>', '', text)
    
    # 4. Decode HTML entities (keep them readable)
    # Already done in data, but just in case
    
    # 5. Replace underscores with spaces for readability
    text = text.replace('_', ' ')
    
    # 6. Keep hashtags as is (they're meaningful in social media)
    # Don't remove # symbol
    
    # 7. Keep numbers (might be meaningful)
    
    # 8. Keep special characters except extreme ones

    # 9. Remove excessive spaces/newlines
    text = ' '.join(text.split())
    
    return text


# Test functions
print("=" * 60)
print("CLEANING FUNCTION COMPARISON")
print("=" * 60)

sample_text = df['post_message'].iloc[0]
print(f"\nOriginal text ({len(sample_text)} chars):")
print(sample_text[:200])

strict = strict_clean(sample_text)
loose = loose_clean(sample_text)

print(f"\nStrict cleaned ({len(strict)} chars):")
print(strict[:200])

print(f"\nLoose cleaned ({len(loose)} chars):")
print(loose[:200])

CLEANING FUNCTION COMPARISON

Original text (1069 chars):
THĂNG CẤP_BẬC HÀM ĐỐI_VỚI 2 CÁN_BỘ , CHIẾN_SỸ HY_SINH Ở ĐÀ_NẴNG Ngày 3/4 , Đại_tướng Tô_Lâm , Bộ_trưởng Bộ Công_an đã ký quyết_định số 2398 / QĐ-BCA-X01 thăng cấp_bậc hàm từ Đại_úy lên Thiếu_tá đối_vớ

Strict cleaned (981 chars):
thăng cấp_bậc hàm đối_với cán_bộ chiến_sỹ hy_sinh ở đà_nẵng ngày đại_tướng tô_lâm bộ_trưởng bộ công_an đã ký quyết_định số qđbcax thăng cấp_bậc hàm từ đại_úy lên thiếu_tá đối_với đồng_chí đặng_thanh_t

Loose cleaned (1069 chars):
THĂNG CẤP BẬC HÀM ĐỐI VỚI 2 CÁN BỘ , CHIẾN SỸ HY SINH Ở ĐÀ NẴNG Ngày 3/4 , Đại tướng Tô Lâm , Bộ trưởng Bộ Công an đã ký quyết định số 2398 / QĐ-BCA-X01 thăng cấp bậc hàm từ Đại úy lên Thiếu tá đối vớ


## 3. Apply Cleaning to Entire Dataset

In [17]:
print("Applying strict cleaning...")
df['text_strict'] = df['post_message'].apply(strict_clean)
print(f"✅ Strict cleaning done")

print("\nApplying loose cleaning...")
df['text_loose'] = df['post_message'].apply(loose_clean)
print(f"✅ Loose cleaning done")

print(f"\nDataset shape: {df.shape}")

Applying strict cleaning...
✅ Strict cleaning done

Applying loose cleaning...
✅ Loose cleaning done

Dataset shape: (4736, 28)


## 4. Quality Check After Cleaning

In [18]:
print("=" * 60)
print("CLEANING QUALITY CHECK")
print("=" * 60)

print(f"\nStrict Cleaning:")
print(f"  Empty texts: {(df['text_strict'] == '').sum()}")
print(f"  Avg length: {df['text_strict'].str.len().mean():.0f} chars")
print(f"  Min length: {df['text_strict'].str.len().min()} chars")
print(f"  Max length: {df['text_strict'].str.len().max()} chars")

print(f"\nLoose Cleaning:")
print(f"  Empty texts: {(df['text_loose'] == '').sum()}")
print(f"  Avg length: {df['text_loose'].str.len().mean():.0f} chars")
print(f"  Min length: {df['text_loose'].str.len().min()} chars")
print(f"  Max length: {df['text_loose'].str.len().max()} chars")

CLEANING QUALITY CHECK

Strict Cleaning:
  Empty texts: 0
  Avg length: 708 chars
  Min length: 3 chars
  Max length: 27797 chars

Loose Cleaning:
  Empty texts: 1
  Avg length: 761 chars
  Min length: 0 chars
  Max length: 28860 chars


## 5. Example Comparisons

In [19]:
print("=" * 60)
print("CLEANING EXAMPLES (5 samples)")
print("=" * 60)

for idx in range(5):
    print(f"\n{'─' * 60}")
    print(f"Sample {idx + 1}:")
    print(f"\nOriginal ({len(df['post_message'].iloc[idx])} chars):")
    print(f"  {df['post_message'].iloc[idx][:150]}...")
    
    print(f"\nStrict ({len(df['text_strict'].iloc[idx])} chars):")
    print(f"  {df['text_strict'].iloc[idx][:150]}...")
    
    print(f"\nLoose ({len(df['text_loose'].iloc[idx])} chars):")
    print(f"  {df['text_loose'].iloc[idx][:150]}...")

CLEANING EXAMPLES (5 samples)

────────────────────────────────────────────────────────────
Sample 1:

Original (1069 chars):
  THĂNG CẤP_BẬC HÀM ĐỐI_VỚI 2 CÁN_BỘ , CHIẾN_SỸ HY_SINH Ở ĐÀ_NẴNG Ngày 3/4 , Đại_tướng Tô_Lâm , Bộ_trưởng Bộ Công_an đã ký quyết_định số 2398 / QĐ-BCA-X...

Strict (981 chars):
  thăng cấp_bậc hàm đối_với cán_bộ chiến_sỹ hy_sinh ở đà_nẵng ngày đại_tướng tô_lâm bộ_trưởng bộ công_an đã ký quyết_định số qđbcax thăng cấp_bậc hàm từ...

Loose (1069 chars):
  THĂNG CẤP BẬC HÀM ĐỐI VỚI 2 CÁN BỘ , CHIẾN SỸ HY SINH Ở ĐÀ NẴNG Ngày 3/4 , Đại tướng Tô Lâm , Bộ trưởng Bộ Công an đã ký quyết định số 2398 / QĐ-BCA-X...

────────────────────────────────────────────────────────────
Sample 2:

Original (7 chars):
  < url >...

Strict (3 chars):
  URL...

Loose (5 chars):
  [URL]...

────────────────────────────────────────────────────────────
Sample 3:

Original (339 chars):
  TƯ_VẤN MÙA_THI : Cách nộp hồ_sơ để trúng_tuyển cao nhất Vào lúc 15 giờ ngày_6.6 , chương_trình Tư_vấn mù

## 6. Save Cleaned Datasets

In [20]:
# Save dataframe with all original columns + 2 cleaned text columns
preprocessed_path = '../../data/preprocessed/preprocessed_data.csv'
df.to_csv(preprocessed_path, index=False)
print(f"✅ Saved to: {preprocessed_path}")
print(f"   Shape: {df.shape}")
print(f"   Total columns: {len(df.columns)}")


✅ Saved to: ../../data/preprocessed/preprocessed_data.csv
   Shape: (4736, 28)
   Total columns: 28


## 8. Final Instructions for Next Steps

In [21]:
print("\n" + "=" * 60)
print("NEXT STEPS - EMBEDDING SELECTION PHASE")
print("=" * 60)

print("""
✅ PREPROCESSED DATA READY:

File: data/preprocessed/preprocessed_data.csv
Structure:
  ├─ text: Original text (raw)
  ├─ text_strict: For TF-IDF, Word2Vec, FastText, GloVe
  ├─ text_loose: For PhoBERT-v2, Bert4News, ChatGPT  
  ├─ label: 0 (Real) or 1 (Fake)
  └─ Features: hour, weekday, day, month, year, time

📊 Embedding Selection Workflow:

1. Load preprocessed_data.csv
2. Stratified train/test split (60/20/20)
   ├─ Train: 2841 samples
   ├─ Val: 947 samples
   └─ Test: 947 samples

3. Test 7 embedding methods on TEST set:
   ├─ TF-IDF → LogReg → AUC
   ├─ Word2Vec → LogReg → AUC
   ├─ FastText → LogReg → AUC
   ├─ GloVe → LogReg → AUC
   ├─ PhoBERT-v2 → LogReg → AUC
   ├─ Bert4News → LogReg → AUC
   └─ ChatGPT text-embedding-3-small → LogReg → AUC

4. Select best embedding for Phase 2 (fine-tuning)

📝 Create: s2_embedding_selection.ipynb
""")



NEXT STEPS - EMBEDDING SELECTION PHASE

✅ PREPROCESSED DATA READY:

File: data/preprocessed/preprocessed_data.csv
Structure:
  ├─ text: Original text (raw)
  ├─ text_strict: For TF-IDF, Word2Vec, FastText, GloVe
  ├─ text_loose: For PhoBERT-v2, Bert4News, ChatGPT  
  ├─ label: 0 (Real) or 1 (Fake)
  └─ Features: hour, weekday, day, month, year, time

📊 Embedding Selection Workflow:

1. Load preprocessed_data.csv
2. Stratified train/test split (60/20/20)
   ├─ Train: 2841 samples
   ├─ Val: 947 samples
   └─ Test: 947 samples

3. Test 7 embedding methods on TEST set:
   ├─ TF-IDF → LogReg → AUC
   ├─ Word2Vec → LogReg → AUC
   ├─ FastText → LogReg → AUC
   ├─ GloVe → LogReg → AUC
   ├─ PhoBERT-v2 → LogReg → AUC
   ├─ Bert4News → LogReg → AUC
   └─ ChatGPT text-embedding-3-small → LogReg → AUC

4. Select best embedding for Phase 2 (fine-tuning)

📝 Create: s2_embedding_selection.ipynb

